# Calibrating X6Y3 — the full process (ZCU216)

The **complete calibration process** of the flux-tunable-qubit walkthrough, minus everything flux,
on the real X6Y3 chip: 8 fixed-frequency qubits in a ring, driven by a ZCU216 running the
`PulseTableSoc`. Spec [software/14](../specs/software/14-full-calibration-x6y3.md) is the audit
that defines the scope.

This is the hardware twin of
[`calibration_process_x6y3_cosim.ipynb`](calibration_process_x6y3_cosim.ipynb) — the same five
stages, the same classes, the same order; only the driver, the shot counts and the qubit count
differ. The co-sim twin is the CI-verified reference for every code path here, so run it first if
anything below misbehaves: it will tell you whether the problem is the chip or the software.

| Stage | What it establishes |
|---|---|
| **1 — readout** | `Punchout` → `ReadoutCalibration` → `Separation` → `Fidelity` → the `Window` knobs → the confusion matrix |
| **2 — frequency** | `Frequency` (Ramsey V-fit) → the `RPEFrequency` polish |
| **3 — one-qubit gates** | `Amplitude` → `Phase` → the `RPEAmplitude`/`RPEPhase` polish → `T1`/`T2` → the EF subspace → the 3-level confusion matrix → DRAG (`optimize_fast_drag` + `Leakage`) |
| **4 — the two-qubit gate** | the drive-form seed → `JAZZ` → `CZAmpFreqSweep` → `CZSweep` → `CZFrequency` → `RelativePhase` → `CZAmplitude` → `LocalPhases` → the `CZRPE` polish → `SpectatorPhase`, on all 8 ring pairs |
| **5 — validation** | the conditionality R at n = 1 vs n = 3 |

Every stage is the same round trip — `Config.from_qcal` → run → fit → `apply()` → `save_qcal` —
through the real qcal tree ([`cal-config-x6y3.yaml`](cal-config-x6y3.yaml)), copied to a working
file so the tree of record is never written in place.

This supersedes [`calibration_x6y3_full.ipynb`](calibration_x6y3_full.ipynb), which covered the
readout / 1Q / EF / two-qubit spine but none of the punchout, window-knob, coherence, 3-level
confusion, DRAG, 2D-CZ-landscape or RPE stages.

In [ ]:
import shutil
from pathlib import Path

import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt

import riscq
from riscq import run as rq
from riscq.cal import (Amplitude, CZAmpFreqSweep, CZAmplitude, CZFrequency, CZRPE, CZSweep,
                       ClassifierN, Config, EFAmplitude, EFFrequency, EFPhase, Fidelity, Frequency,
                       JAZZ, Leakage, LocalPhases, Phase, Punchout, RPEAmplitude, RPEFrequency,
                       RPEPhase, ReadoutCalibration, ReadoutFidelity, RelativePhase, Separation,
                       SpectatorPhase, T1, T2, Window, calc_cz_frequency, kernels,
                       optimize_fast_drag, pair_key, rcorr)
from riscq.cal.base import (GATE_CH, SEP, acquire_shots, batch_timeout, ef_table, grid_period,
                            readout_tables, relax_batches, x90_vz)
from riscq.cal.drag import ef_spectral_weight
from riscq.cal.readout import _rawiq_prog
from riscq.cal.rpe import CZ_TARGETS
from riscq.cal.twoqubit import _cz_cond_R, _cz_freq_word
from riscq.driver.remote import RemoteDriver, upload_bundle
from riscq.lang import Array, compile_kernel
from riscq.map import LEAD, SocMap, SocParams
from riscq.pulses import units

MHz, GHz, us = 1e6, 1e9, 1e-6

## Connect to the board and the config of record

The chassis is the ZCU216 board server ([docs/software/board-server.md](../docs/software/board-server.md))
with an X6Y3 gateware bundle loaded. The config is the real qcal tree, copied to a **working file**
that every cell reloads and every applied proposal is saved back into.

In [ ]:
BOARD = '192.168.1.122'                   # the ZCU216's LAN address (or the full PYRO: uri)
PORT = 9091

SW = Path(riscq.__file__).resolve().parents[1]              # .../software
SRC = SW.parent / 'examples' / 'cal-config-x6y3.yaml'       # the real X6Y3 qcal tree
WORK = SW / 'build' / 'x6y3_process_config.yaml'            # the working copy
WORK.parent.mkdir(exist_ok=True)
shutil.copy(SRC, WORK)

drv = RemoteDriver(BOARD, PORT)
print('server:', drv.board.info())

# first time only — push the X6Y3 build up and load it:
# upload_bundle(drv, 'x6y3', xsa='../build/x6y3/top.xsa',
#               params_json='../software/configs/x6y3.json')
# info = drv.board.load('x6y3')
# assert info['mts_result'] == 0, 'multi-tile sync missed its target latencies'

m = SocMap(SocParams.from_json(drv.board.get_params()))
QUBITS = list(range(8))
PAIRS = [(0, 1), (1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (6, 7), (7, 0)]   # the ring
SANDWICH = [(5, 6), (6, 7)]                    # EF-shelved pairs (q6 is the shelf, spec 04 §1)
PLAIN = [p for p in PAIRS if p not in SANDWICH]

cfg = Config.from_qcal(WORK)
cfg.check_hardware(m.params)     # the tree's DAC/ADC rates + interpolation must describe THIS bundle
print(f"connected to '{m.params.name}': {m.params.qubit_num} cores;  herald = {cfg['readout/herald']}")

def step(cal):
    '''run -> print -> apply if ok -> persist to the working tree (qcal's auto write-back).'''
    r = cal.run(drv)
    print(f'{r.label}: ok={r.ok}  proposal={r.proposal}')
    if r.ok:
        r.apply()
        r.cfg.save_qcal(WORK)
    else:
        print('  fit failed — config left unchanged (fail-loud, spec 13 §2)')
    return r

# Stage 1 — readout

Nothing else can be measured until a shot can be classified. Find the resonator and a drive power
that resolves it, train the discriminator, then tune each readout knob against a metric the
discriminator itself defines. All 8 qubits run simultaneously on the frequency-multiplexed readout.

## Punchout — the frequency × drive-power map

The resonator pulls with drive power: weak tones see the dressed (qubit-coupled) frequency, strong
tones saturate it back toward the bare cavity. Read the map and pick a power **below** the
punch-through — that is the amplitude `Separation` and `Fidelity` refine from. This proposes
nothing; it is a bring-up tool, like the wideband VNA.

In [ ]:
cfg = Config.from_qcal(WORK)
po = Punchout(cfg, QUBITS, amps=(0.05, 0.1, 0.2, 0.4, 0.7), span=6 * MHz, points=41,
              shots=32).run(drv)

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for q, ax in zip(QUBITS, axes.flat):
    d = po.data[q]
    for i, a in enumerate(d['amps']):
        ax.plot((d['x'] - d['x'].mean()) / MHz, d['mag'][i], '-', label=f'{a:g}')
    ax.set_title(f'q{q}'); ax.set_xlabel('readout freq − centre [MHz]'); ax.set_ylabel('|IQ|')
    ax.legend(fontsize=6, title='amp')
plt.tight_layout(); plt.show()

In [ ]:
cfg = Config.from_qcal(WORK)
rc = step(ReadoutCalibration(cfg, QUBITS, shots=1000, gate='X90'))

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for q, ax in zip(QUBITS, axes.flat):
    d = rc.data[q]
    ax.scatter(d['iq0'][:, 0], d['iq0'][:, 1], s=4, label='|0>')
    ax.scatter(d['iq1'][:, 0], d['iq1'][:, 1], s=4, label='|1>')
    ax.set_title(f"q{q}  sep={d['separation']:.2f}")
    ax.set_aspect('equal'); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
cfg = Config.from_qcal(WORK)
sep = step(Separation(cfg, QUBITS, span=2.5 * MHz, points=31, shots=32, gate='X90'))
for q in QUBITS:
    plt.plot((sep.data[q]['x'] - sep.data[q]['x'].mean()) / MHz, sep.data[q]['y'], '-', label=f'q{q}')
plt.xlabel('readout freq − centre [MHz]'); plt.ylabel('cluster SNR'); plt.legend(ncol=4); plt.show()

cfg = Config.from_qcal(WORK)
fid = step(Fidelity(cfg, QUBITS, amp_span=0.005, points=31, shots=3000, gate='X90'))
for q in QUBITS:
    plt.plot(fid.data[q]['x'], fid.data[q]['y'], '-', label=f'q{q}')
plt.xlabel('readout amp'); plt.ylabel('confusion diagonal'); plt.legend(ncol=4); plt.show()

## The window knobs — how long to listen, and when to start

Three timings the walkthrough tunes: how long the integrator is open (`demod/dur`), how long the
drive tone plays (`dur`), and how long after the drive starts the integrator opens (`demod/delay`,
the cable delay plus the cavity ring-up). None is an on-core sweep — the two durations are readout
table-slot fields, the delay is a per-run kernel param — so `Window` reruns the measurement once per
value and takes the argmax of the confusion diagonal, the metric `Fidelity` scores.

In [ ]:
for knob, durs in (('demod/dur', (2e-7, 4e-7, 6e-7, 8e-7, 1.2e-6)),
                   ('dur', (4e-7, 6e-7, 8e-7, 1.2e-6, 1.6e-6)),
                   ('demod/delay', (0.0, 5e-8, 1e-7, 2e-7, 3e-7))):
    cfg = Config.from_qcal(WORK)
    w = step(Window(cfg, QUBITS, durs=durs, shots=200, gate='X90', knob=knob))
    plt.figure(figsize=(5, 3))
    for q in QUBITS:
        plt.plot(np.array(w.data[q]['x']) * 1e9, w.data[q]['y'], 'o-', label=f'q{q}')
    plt.xlabel(f'{knob} [ns]'); plt.ylabel('confusion diagonal'); plt.legend(ncol=4, fontsize=6)
    plt.title(knob); plt.show()

In [ ]:
cfg = Config.from_qcal(WORK)
rof = step(ReadoutFidelity(cfg, QUBITS, shots=5000, gate='X90'))
for q in QUBITS:
    print(f"q{q} confusion (row = prepared, col = classified):")
    print(np.round(rof.data[q]['confusion'], 3), f"  fidelity={rof.data[q]['fidelity']:.3f}")

# Stage 2 — the qubit frequency

`Frequency` runs Ramsey fringes at the reference's ±2.5/±5 MHz applied detunings and fits qcal's
`a·|x − b| + c` V; the vertex is the applied detuning that cancels the tree's error, so the
correction carries a sign a magnitude-only fit could not give.

Coherence (`T1`/`T2`) is measured at the end of stage 3 rather than here, because both need a π
pulse that works — the one place this notebook's order departs from the walkthrough's, and it
departs the way a real bring-up does.

In [ ]:
cfg = Config.from_qcal(WORK)
detunings = np.array([-5, -2.5, 2.5, 5]) * MHz          # the reference's exact set
fr = step(Frequency(cfg, QUBITS, detunings=detunings, t_max=1 * us, points=30, shots=512))

for q in QUBITS:
    plt.plot(fr.data[q]['applied'], fr.data[q]['obs'], 'o-', label=f'q{q}')
plt.xlabel('applied detuning [codes]'); plt.ylabel('|fringe| [codes]'); plt.legend(ncol=4); plt.show()
for q in QUBITS:
    print(f"qubit/{q}/freq -> {cfg[f'qubit/{q}/freq'] / GHz:.6f} GHz")

## The RPE polish

`Frequency`'s precision is bounded by the shot noise on one fringe. **Robust phase estimation**
repeats the idle `d` times so a residual carrier error writes `d` times the phase, and walks an
exponential ladder of `d`, resolving each rung's `2π/d` ambiguity with the previous rung's
estimate — precision then improves like `1/d` rather than `1/√shots` (spec 14 F5).

Two things govern the knobs. `t_idle` must be **well above the gate length**: every rung is
bracketed by the same two X90s, and the detuning writes phase during those too, biasing the
estimate by roughly `t_pulse/(d·t_idle)`. And the depth-1 rung is only unambiguous while the
residual is under `1/(2·t_idle)`, which is why RPE runs *after* `Frequency` and not instead of it.
Check the reported contrast: if it does not stay O(1) across the ladder, the ladder is deeper than
the coherence time and the trusted depth will have been cut back.

In [ ]:
cfg = Config.from_qcal(WORK)
T_IDLE = 2e-6            # >> the 35 ns X90, and unambiguous out to 1/(2·t_idle) = 250 kHz
cal = RPEFrequency(cfg, QUBITS, t_idle=T_IDLE, depths=(1, 2, 4, 8, 16), shots=512)
rpe = step(cal)          # step() returns the Result; the ladder is on `cal`

for q in QUBITS:
    a = cal.angles[q]
    plt.plot(a.depths, a.estimates['Z'], 'o-', label=f'q{q}')
    print(f"q{q}: detuning {cal.recovered_detuning[q] / 1e3:+8.2f} kHz  "
          f"+-{a.uncertainty / (2 * np.pi * T_IDLE) / 1e3:.2f}  "
          f"last good depth={a.last_good_depth}  contrast={np.round(a.contrast, 2).tolist()}")
plt.xscale('log', base=2); plt.xlabel('depth'); plt.ylabel('phase per idle step [rad]')
plt.legend(ncol=4); plt.show()

# Stage 3 — one-qubit gates

`Amplitude` coarse (the full-range Rabi cosine) then a narrow relative pass with 4 X90s to amplify
what is left; `Phase` fixes the axis by the line crossing of qcal's two Rz-decorated sequences.

In [ ]:
cfg = Config.from_qcal(WORK)
ac = step(Amplitude(cfg, QUBITS, gate='X90', n_gates=1, amp_span=(0.03, 0.97), points=31, shots=512))

cfg = Config.from_qcal(WORK)
af = step(Amplitude(cfg, QUBITS, gate='X90', n_gates=4, amp_span=(0.7, 1.3), relative_amp=True,
                    points=31, shots=512))
for q in QUBITS:
    print(f"q{q}: X90 amp -> {cfg[f'qubit/{q}/x90/amp']:.5f}")

In [ ]:
cfg = Config.from_qcal(WORK)
for q in QUBITS:                                 # the reference's per-qubit relative loop
    step(Phase(cfg, [q], span=0.25, points=21, shots=512, relative_phase=True))

cfg = Config.from_qcal(WORK)
ph = step(Phase(cfg, [1, 3, 5, 7], span=0.3, points=31, shots=512))   # the wider absolute pass
for q in QUBITS:
    vz = cfg.get(f'qubit/{q}/x90/vz', [0.0, 0.0])
    print(f'q{q}: X90 virtual-Z pair -> [{vz[0]:+.4f}, {vz[1]:+.4f}] rad')

## The X180 — the other pulse, and its own axis

The reference calibrates the π everywhere it calibrates the π/2 (GE and EF, ladders to n_gates = 96),
so this notebook does too. `Amplitude(gate='X')` is the same Rabi cosine against the config's own `x`
pulse, targeting a full π — the repetition guard becomes n % 2 rather than n % 4. On X6Y3 the X is
the X90 at **double length**, so the fine pass should barely move it.

`Phase(gate='X')` is *not* the X90 measurement re-pointed (spec 14 §3.3). It is one circuit,
`X90 · X · X90` — a 2π rotation that returns to |0⟩ only when the X sits on the X90s' axis — swept
over the X's own **axis** phase (`qubit/{q}/x/phase`, the FAST_DRAG's `phase` kwarg), not a
virtual-Z pair. The fringe runs at 2φ, so the recovered axis is defined mod π: both solutions are
the same gate.

In [ ]:
cfg = Config.from_qcal(WORK)
xa = step(Amplitude(cfg, QUBITS, gate='X', n_gates=1, amp_span=(0.03, 0.97), points=31, shots=512))

cfg = Config.from_qcal(WORK)
xf = step(Amplitude(cfg, QUBITS, gate='X', n_gates=4, amp_span=(0.7, 1.3), relative_amp=True,
                    points=31, shots=512))

cfg = Config.from_qcal(WORK)
xp = step(Phase(cfg, QUBITS, gate='X', points=31, shots=512))
for q in QUBITS:
    print(f"q{q}: X amp -> {cfg[f'qubit/{q}/x/amp']:.5f}   "
          f"X axis phase -> {float(cfg.get(f'qubit/{q}/x/phase', 0.0)):+.4f} rad")

## The RPE polish on the gate

The same amplification argument applied to the gate: repeating the X90 `d` times multiplies its
rotation error `d`-fold. `RPEAmplitude` recovers the per-gate rotation angle Θ and scales the
amplitude by `(π/2)/Θ`.

Each rung reads four trains — `d`, `d+1`, `d+2`, `d+3`. Two X90s make a π rotation, so the `d+2`
train returns the exact opposite outcome to the `d` train; that pairing is what stops a readout
fringe not centred on ½ (T1 decay before the latch, or an asymmetric discriminator) from leaking
into the angle. The trains are paced against the depth-4 posted-link queue (spec 14 F1), so the
deep rungs actually play every gate.

In [ ]:
cfg = Config.from_qcal(WORK)
cal = RPEAmplitude(cfg, QUBITS, depths=(1, 2, 4, 8, 16), shots=512)
ra = step(cal)

for q in QUBITS:
    a = cal.angles[q]
    plt.plot(a.depths, a.estimates['X'], 'o-', label=f'q{q}')
    print(f"q{q}: rotation {cal.recovered_angle[q]:.5f} rad (target {np.pi / 2:.5f}, "
          f"err {1e2 * (cal.recovered_angle[q] / (np.pi / 2) - 1):+.2f}%)  "
          f"amp -> {cfg[f'qubit/{q}/x90/amp']:.5f}  last good depth={a.last_good_depth}")
plt.axhline(np.pi / 2, color='k', ls=':')
plt.xscale('log', base=2); plt.xlabel('depth'); plt.ylabel('rotation per X90 [rad]')
plt.legend(ncol=4); plt.show()

## The RPE polish on the gate's axis

The **interleaved** echo `Z90·X90·X90·Z90·Z90·X90·X90·Z90`, repeated `d` times, cancels the rotation
*angle* and amplifies the tilt of the rotation axis out of the drive plane; paired with the direct
train it recovers both the rotation (`X`) and the tilt (`Z`) at once — qcal's linearized estimator.

What it can see is worth stating precisely: a uniform pulse `phase` on every X90 alike conjugates
the whole sequence by an Rz, which a z-in/z-out measurement is blind to by construction — a frame
convention, not an error. What the echo measures is the frame advance *between* gates being wrong
for the phase the pulse leaves behind: an ac-Stark shift, or a mis-set virtual-Z pair. The
correction is the symmetric split of the measured composite (≈0.64·Z on a quarter turn, not Z/2)
and is added to **both** slots of the pair, since only the pair's sum is what a repeated gate feels.

A depth-`d` rung spends `4d+3` X90s on the paced train grid, so keep the ladder inside T1.

In [ ]:
cfg = Config.from_qcal(WORK)
cal = RPEPhase(cfg, QUBITS, depths=(1, 2, 4, 8), shots=512)
rph = step(cal)

for q in QUBITS:
    a = cal.angles[q]
    plt.plot(a.depths, a.estimates['Z'], 'o-', label=f'q{q}')
    print(f"q{q}: axis tilt {cal.recovered_tilt[q]:+.5f} rad  -> vz "
          f"[{cfg[f'qubit/{q}/x90/vz'][0]:+.5f}, {cfg[f'qubit/{q}/x90/vz'][1]:+.5f}]   "
          f"last good depth={a.last_good_depth}  contrast={np.round(a.contrast, 2).tolist()}")
plt.axhline(0.0, color='k', ls=':')
plt.xscale('log', base=2); plt.xlabel('depth'); plt.ylabel('axis tilt Z [rad]')
plt.legend(ncol=4); plt.show()

## Coherence — T1 and T2*

With a working π, the two decay constants: `T1` preps |1⟩ and sweeps the delay before the readout;
`T2` is a Ramsey at a deliberate detuning whose fringe envelope decays at T2*. Both bound every
depth used above and below — an RPE ladder or a gate train taken past them measures noise.

In [ ]:
cfg = Config.from_qcal(WORK)
t1 = step(T1(cfg, QUBITS, points=15, shots=512))
for q in QUBITS:
    plt.plot(t1.data[q]['x'], t1.data[q]['y'], 'o-', label=f'q{q}')
plt.xlabel('delay [batches]'); plt.ylabel('P(|1>)'); plt.legend(ncol=4); plt.show()

cfg = Config.from_qcal(WORK)
t2 = step(T2(cfg, QUBITS, detune=500e3, points=25, t0=4e-8, dt=4e-7, shots=512))
for q in QUBITS:
    plt.plot(t2.data[q]['x'], t2.data[q]['y'], 'o-', label=f'q{q}')
plt.xlabel('Ramsey wait [batches]'); plt.ylabel('P(|1>)'); plt.legend(ncol=4); plt.show()

for q in QUBITS:
    print(f"q{q}: T1 -> {cfg[f'qubit/{q}/T1'] * 1e6:6.2f} us   "
          f"T2* -> {cfg[f'qubit/{q}/T2'] * 1e6:6.2f} us")

## The EF subspace

The |1⟩→|2⟩ manifold above the computational one — needed because leakage into |2⟩ is what DRAG
suppresses, and because the (5, 6)/(6, 7) sandwich CZ pairs shelve through it. The hardware `res`
bit is 2-level, so P(|2⟩) is read host-side through a 3-level `ClassifierN` trained from three RAW
reference clouds per qubit: |0⟩ and |1⟩ through the 2-level RAW program, |2⟩ through a GE π
followed by the tree's **stored** EF X. All captures run in the demod zero-phase frame, the frame
`ReadoutCalibration` trains in.

In [ ]:
cfg = Config.from_qcal(WORK)

def train_classifier3(qubits, shots=400):
    '''{q: ClassifierN} from |0>/|1>/|2> RAW reference clouds (all qubits in parallel per level).'''
    clouds = {q: [] for q in qubits}
    progs, timeout = {}, 0
    for q in qubits:                              # |0> and |1>: the 2-level RAW program (prep scalar)
        prog, period = _rawiq_prog(m, cfg, q, 'X90', shots)
        progs[q] = prog
        timeout = max(timeout, batch_timeout(shots * period))
    rq.setup(drv, m, progs)
    for level in (0, 1):
        iq = acquire_shots(drv, m, progs, level, shots, timeout)
        for q in qubits:
            clouds[q].append(iq[q])
    progs, timeout = {}, 0
    for q in qubits:                              # |2>: GE pi then the STORED EF X, captured RAW
        table, ge_freq, ef_freq = ef_table(cfg, q, m, 'x')
        ro, demod, code, dur, ddly = readout_tables(cfg, q, m, phase=0.0)   # the zero-phase frame
        ge = table.pulses['x90'].dur_batches(m, GATE_CH)
        ef = table.pulses['ef'].dur_batches(m, GATE_CH)
        period = grid_period(relax_batches(cfg, m), SEP + ef + LEAD + 2 * ge, dur, ddly)
        progs[q] = compile_kernel(kernels.k_ef_rabi, m, tables=dict(gate=table, ro=ro, demod=demod),
                                  out=Array(2 * shots), npts=1, shots=shots, period=period,
                                  ngates=1, code=code, ddly=ddly, ge_freq=ge_freq, ef_freq=ef_freq,
                                  **x90_vz(cfg, q))
        timeout = max(timeout, batch_timeout(shots * period))
    par = {q: {'a0q': units._amp_code(float(cfg[f'qubit/{q}/EF/x/amp'])) << 16, 'daq': 0}
           for q in qubits}
    out = rq.run(drv, m, progs, params=par, results=['out'], timeout=timeout)
    for q in qubits:
        clouds[q].append(out[q]['out'].reshape(shots, 2).astype(float))
    return {q: ClassifierN(clouds[q]) for q in qubits}

CLF = train_classifier3(QUBITS)
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for q, ax in zip(QUBITS, axes.flat):
    for lvl, c in enumerate(CLF[q].clusters):
        ax.scatter(c[:, 0], c[:, 1], s=4, label=f'|{lvl}>')
    ax.set_title(f'q{q}  sep={CLF[q].separation:.2f}')
    ax.set_aspect('equal'); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
cfg = Config.from_qcal(WORK)
efr = step(EFFrequency(cfg, QUBITS, CLF, detune=5 * MHz, n_detune=4, points=14, shots=48))
for q in QUBITS:
    print(f"qubit/{q}/EF/freq -> {cfg[f'qubit/{q}/EF/freq'] / GHz:.6f} GHz")

cfg = Config.from_qcal(WORK)
efa = step(EFAmplitude(cfg, QUBITS, CLF, gate='X90', n_gates=1, points=21, shots=48))
cfg = Config.from_qcal(WORK)
efx = step(EFAmplitude(cfg, QUBITS, CLF, gate='X', n_gates=1, points=21, shots=48))
for q in QUBITS:
    print(f"q{q}: EF x90 amp -> {cfg[f'qubit/{q}/EF/x90/amp']:.5f}   "
          f"EF x amp -> {cfg[f'qubit/{q}/EF/x/amp']:.5f}")

cfg = Config.from_qcal(WORK)
efp = step(EFPhase(cfg, QUBITS, CLF, points=21, span=0.25, shots=48, relative_phase=True))
for q in QUBITS:
    vz = cfg[f'qubit/{q}/EF/x90/vz']
    print(f'q{q}: EF X90 virtual-Z pair -> [{vz[0]:+.4f}, {vz[1]:+.4f}] rad')

### The 3-level confusion matrix and `rcorr`

`ReadoutFidelity(n_levels=3)` measures the full 3×3 matrix over RAW shots at the |0⟩/|1⟩/|2⟩ preps.
`rcorr(P, cmat)` inverts it, turning a population read through an imperfect discriminator back into
the population the qubit actually had — which is what makes the leakage numbers below honest
(spec 14 F2). The matrix also lands in the tree at `readout/{q}/cmat`.

It runs **after** the EF cals: its |2⟩ row is a real prep — a GE π followed by the config's EF X —
so it wants that pulse freshly calibrated, not the stored one the classifier was trained against.

In [ ]:
cfg = Config.from_qcal(WORK)
rof3 = ReadoutFidelity(cfg, QUBITS, shots=2000, gate='X90', n_levels=3, classifier=CLF).run(drv)
rof3.apply()
rof3.cfg.save_qcal(WORK)

CMAT = {}
for q in QUBITS:
    CMAT[q] = rof3.data[q]['confusion']
    print(f'q{q} 3-level confusion (fidelity {rof3.data[q]["fidelity"]:.3f}):')
    print(np.round(CMAT[q], 3))

## DRAG — shaping the pulse away from the EF transition

Leakage out of the computational subspace is driven by the pulse's own spectral content **at the
EF transition**, which the FAST_DRAG envelope's `N` and `weights` shape. `optimize_fast_drag` is
qcal's `optimize_FAST_DRAG`: a host-side coordinate descent minimising `|FFT(envelope)|` at that
frequency. No hardware is involved — the envelope and its transform are the entire measurement.

`Leakage` is the measured half: a 101×X90 train read for P(|2⟩) through the 3-level classifier,
once per value of one swept config path, taking the value that minimises the leaked population.
The reference sweeps the X90's virtual-Z phases first, then `N` and `weights/0` — and it runs the
vz sweep in its *diagnostic* direction (`maximize=True`, amplifying the leakage to make it
visible); this notebook takes the calibration direction, the class default. Only `weights/0` is
actually swept on hardware there: the reference's `N` cell is shadowed dead code, and `N` comes
from the host FFT alone (spec 14 §1). Each point recompiles (a vz pair is a kernel binding, a
kwarg changes the envelope image), so keep the value lists short.

**This is the one stage the co-sim twin cannot check.** Its `ThreeLevelModel` rotates only the
stronger of the two transitions, so a GE drive never populates |2⟩ and there is no leakage to
plant — the numbers below are the first real test of this code path (spec 14 F3).

In [ ]:
cfg = Config.from_qcal(WORK)
for q in QUBITS:
    before = ef_spectral_weight(cfg, q, m)
    best = optimize_fast_drag(cfg, q, m)[f'qubit/{q}/x90/kwargs']
    after = ef_spectral_weight(cfg, q, m, 'x90', best)
    print(f"q{q}: |FFT| at f_EF {before:.4g} -> {after:.4g}  "
          f"N -> {best['N']}, weights -> {best['weights']}")
    cfg[f'qubit/{q}/x90/kwargs'] = best
cfg.save_qcal(WORK)

In [ ]:
# the reference's order: the vz phases first, then the FAST_DRAG N. Each value is a recompile.
cfg = Config.from_qcal(WORK)
for q in QUBITS:
    vz0 = float(cfg[f'qubit/{q}/x90/vz'][0])
    values = [[vz0 + d, vz0 + d] for d in (-0.04, -0.02, 0.0, 0.02, 0.04)]
    lk = step(Leakage(cfg, [q], CLF, f'qubit/{q}/x90/vz', values, n_gates=101, shots=256))
    plt.plot([v[0] for v in values], lk.data[q]['y'], 'o-', label=f'q{q}')
plt.xlabel('X90 virtual-Z phase [rad]'); plt.ylabel('P(|2>) after 101 X90s')
plt.legend(ncol=4, fontsize=7); plt.show()

# Stage 4 — the CZ

X6Y3 is fixed-frequency with no coupler, so the CZ is the **drive form**: both qubits' own drive
lines play a tone at a shared frequency near the |11⟩↔|02⟩ resonance. `calc_cz_frequency` seeds
that frequency at `(f₁₁ + f₀₂)/4` from the 1Q spectrum just calibrated; on this already-calibrated
tree the stored values are better, so the cell only prints the comparison (the sandwich pairs
activate in the shelved manifold, so their seed is expected to sit elsewhere).

In [ ]:
cfg = Config.from_qcal(WORK)
probe = cfg.copy()
calc_cz_frequency(probe, PAIRS, state='02', form='drive')
for pair in PAIRS:
    pk = pair_key(pair)
    have, seed = cfg[f'two_qubit/{pk}/CZ/freq'] / GHz, probe[f'two_qubit/{pk}/CZ/freq'] / GHz
    tag = '  (shelved manifold — seed not applicable)' if pair in SANDWICH else ''
    print(f'{pk}: config {have:.4f} GHz   seed {seed:.4f} GHz   d {1e3 * (seed - have):+7.1f} MHz{tag}')

## JAZZ — the always-on ZZ around the ring

The BIRD-echo Ramsey: on a fixed-frequency chip there is no null knob, so JAZZ **characterizes**
the residual ZZ per pair rather than calibrating it. Since F0 the pair key round-trips into the
artefact, so `ZZ11` now survives `save_qcal`.

In [ ]:
ZZ = {}
for pair in PAIRS:
    cfg = Config.from_qcal(WORK)
    r = step(JAZZ(cfg, pair, detune=2 * MHz, points=20, shots=120))
    ZZ[pair] = r.proposal.get(f'two_qubit/{pair_key(pair)}/ZZ11', float('nan'))
for pair in PAIRS:
    print(f'{pair}: ZZ11 = {ZZ[pair] / 1e3:+8.1f} kHz')

## The 2D landscape — amplitude × frequency

Before walking the 1D chain, map the neighbourhood: `CZAmpFreqSweep` sweeps the drive amplitude
(lockstep on both lines) against the drive frequency and scores each point by the same
conditionality R the 1D cals use. The argmax is where the chain should start, and the shape of the
ridge tells you whether the 1D cuts below are being taken through a real optimum (spec 14 F4).

In [ ]:
for pair in PAIRS:
    cfg = Config.from_qcal(WORK)
    afs = step(CZAmpFreqSweep(cfg, pair, amps=np.linspace(0.2, 0.6, 9), span=8 * MHz,
                              points=15, ngates=1, shots=120))
    d, best = afs.data[pair], afs.fit[pair]
    plt.figure(figsize=(4.5, 3))
    plt.pcolormesh(d['freqs'] / GHz, d['amps'], d['R'], shading='nearest')
    plt.colorbar(label='R'); plt.xlabel('CZ freq [GHz]'); plt.ylabel('line amp')
    plt.title(f'{pair}: argmax amp {best["amp"]:.3f}, f {best["freq"] / GHz:.4f} GHz')
    plt.show()

## The 1D chain — plain pairs

In the spec-04 order, each step refining one knob at the previous step's optimum:
`CZSweep('freq')` (the |11⟩-population dip re-pins the resonance) → `CZFrequency`
(conditionality-R argmax) → `RelativePhase` (the target line's tone phase — the two lines sum
coherently, so R peaks where the round trip closes) → `CZAmplitude` (the R error-amplification
ladder) → `LocalPhases` (the single-qubit Z each CZ leaves behind).

`RelativePhase` sweeps a **narrow** window around the stored phase, as the reference does: R vs
phase is a cosine, and over a full turn the parabola fit fails and the class falls back to the bare
sweep-point argmax (spec 14 §2).

In [ ]:
for pair in PLAIN:
    print(f'=== pair {pair} ===')
    cfg = Config.from_qcal(WORK)
    step(CZSweep(cfg, pair, 'freq', span=10 * MHz, points=21, shots=120))
    cfg = Config.from_qcal(WORK)
    step(CZFrequency(cfg, pair, span=6 * MHz, points=15, ngates=1, shots=120))
    cfg = Config.from_qcal(WORK)
    step(RelativePhase(cfg, pair, span=3.0, points=15, ngates=1, shots=120))
    cfg = Config.from_qcal(WORK)
    step(CZAmplitude(cfg, pair, n_gates=(1, 3, 5), window=0.3, points=11, shots=120))
    cfg = Config.from_qcal(WORK)
    step(LocalPhases(cfg, pair, points=15, shots=120))

## The 1D chain — the EF-sandwich pairs (5, 6) and (6, 7)

**Prerequisite (satisfied above):** q6's `EF/freq` and `EF/x/amp`. The sandwich brackets its two
drive tones with the string-reference pre/post-pulse `single_qubit/6/EF/X/pulse` (shelve |1⟩→|2⟩,
un-shelve after) and the classes resolve it internally — the chain itself is identical.

In [ ]:
for pair in SANDWICH:
    print(f'=== pair {pair} (EF sandwich, shelf q6) ===')
    cfg = Config.from_qcal(WORK)
    step(CZSweep(cfg, pair, 'freq', span=10 * MHz, points=21, shots=120))
    cfg = Config.from_qcal(WORK)
    step(CZFrequency(cfg, pair, span=6 * MHz, points=15, ngates=1, shots=120))
    cfg = Config.from_qcal(WORK)
    step(RelativePhase(cfg, pair, span=3.0, points=15, ngates=1, shots=120))
    cfg = Config.from_qcal(WORK)
    step(CZAmplitude(cfg, pair, n_gates=(1, 3, 5), window=0.3, points=11, shots=120))
    cfg = Config.from_qcal(WORK)
    step(LocalPhases(cfg, pair, points=15, shots=120))

## The RPE polish on the CZ

The amplification argument applied to the two-qubit gate. `CZRPE` runs three conditional-Ramsey
ladders — the target's fringe with the control held |0⟩ and |1⟩, and (roles swapped) the control's
fringe with the target held |1⟩ — and inverts the three accumulated angles into the CZ's
**generator** angles `ZZ`, `IZ`, `ZI`, whose ideal values for `diag(1, 1, 1, −1)` are
`(−π/2, +π/2, +π/2)`.

Every rung is read at **both close signs** in both quadratures, which divides the fringe centre out
of the angle and makes the Ramsey qubit's own marginal a valid estimator — no joint two-qubit
post-selection (spec 14 F5 finding 5; the reference post-selects joint bitstrings and still assumes
a centred fringe).

**`ZZ` is the number to read**: it is the conditional phase the whole chain above was calibrating,
and its residual is what a large re-run of `CZFrequency`/`CZAmplitude` would drive to zero. The
class never writes it — that knob needs a measured slope, i.e. qcal's `LinearResponse`/optimizer
stack, out of scope by spec 14 §4.

`IZ`/`ZI` are the local phases, and the class *does* propose a damped `+=` on the pair's virtual-Z
entries. This notebook **reports rather than applies** it: in the co-sim twin the generator's split
of the pair's local phase between the two qubits differs from the frame `LocalPhases` writes (their
frame-independent sum agrees), so the polish is left as an explicit operator decision — uncomment
the `apply` once the residual signs check out against your frame convention. Watch the contrast: a
CZ ladder is long, and the trusted depth is cut back the moment it leaves the pair's coherence.

In [ ]:
for pair in PAIRS:
    cfg = Config.from_qcal(WORK)
    cal = CZRPE(cfg, pair, depths=(1, 2, 4, 8), shots=256)
    r = cal.run(drv)                     # REPORTED, not applied — see above
    a = cal.angles.get(pair)
    if a is None:
        print(f'{pair}: no angles — {r.data[pair].get("error", "")}')
        continue
    print(f'{pair}: ' + '  '.join(f'{n} {a.trusted[n]:+.4f} (residual {a.trusted_error[n]:+.4f})'
                                  for n in ('ZZ', 'IZ', 'ZI'))
          + f'   IZ+ZI {a.trusted["IZ"] + a.trusted["ZI"]:+.4f} (ideal {np.pi:.4f})'
          + f'   last good depth={a.last_good_depth}'
          + f'   contrast={np.round(a.contrast, 2).tolist()}')
    # to accept the local-phase polish on a pair, once its residual signs check out in your frame:
    #   r.apply(); r.cfg.save_qcal(WORK)

## Spectator phases

A pair's CZ also kicks the frame of ring neighbours that sit in its pulse list as extra virtual-Z
entries. `SpectatorPhase` runs the bystander's Ramsey on a third core around one CZ fire and writes
the spectator's channel-matched entry — the wrap-aware plain mean, with no conditional-π removal,
because the spectator is outside the gate. Spectators are read off the pulse list itself.

In [ ]:
def spectators(cfg, pair):
    '''The ring neighbours with a virtual-Z entry in this pair's CZ pulse list.'''
    out = []
    for p in cfg[f'two_qubit/{pair_key(pair)}/CZ/pulse']:
        if not isinstance(p, str) and p.get('env') == 'virtualz':
            q = int(str(p['channel']).split('.')[0][1:])
            if q not in pair:
                out.append(q)
    return out

for pair in PAIRS:
    cfg = Config.from_qcal(WORK)
    for s in spectators(cfg, pair):
        cfg = Config.from_qcal(WORK)
        step(SpectatorPhase(cfg, pair, spectator=s, points=15, shots=120))

# Stage 5 — validation

The walkthrough validates with randomised benchmarking; that needs a licensed dependency the
project does not carry (spec 14 §4), so the check here is **error amplification**, which resolves
the same coherent errors: R at n = 1 must survive n = 3 CZs, because three gates compound any
residual frequency, amplitude or phase error into a visible drop. Re-running `JAZZ` after the chain
gives the second handle — the residual ZZ the calibrated gate leaves behind.

In [ ]:
for pair in PAIRS:
    cfg = Config.from_qcal(WORK)
    row = []
    for n in (1, 3):
        R, _ = _cz_cond_R(cfg, drv, m, pair, 'freq', _cz_freq_word(cfg, pair, m), 0, 1, n, 120)
        row.append(f'R(n={n})={float(R[0]):.3f}')
    print(f'{pair}: ' + '  '.join(row))

## What the process wrote

The calibrated working tree against the pristine config of record — every path the process moved,
through the full `from_qcal`/`save_qcal` round trip. Diff `WORK` against `cal-config-x6y3.yaml` to
see it in the artefact itself.

In [ ]:
def flat(c, prefix='', out=None):
    out = {} if out is None else out
    for k, v in (c if isinstance(c, dict) else c.to_dict()).items():
        p = f'{prefix}/{k}' if prefix else str(k)
        if isinstance(v, dict):
            flat(v, p, out)
        else:
            out[p] = v
    return out

before, after = flat(Config.from_qcal(SRC)), flat(Config.from_qcal(WORK))
changed = [p for p in sorted(after) if after[p] != before.get(p)]
print(f'{len(changed)} paths changed:')
for p in changed:
    b, a = before.get(p), after[p]
    if isinstance(a, float) and isinstance(b, float):
        print(f'  {p}: {b:.6g} -> {a:.6g}')
    else:
        print(f'  {p}: {b} -> {a}')

## Disconnect

In [ ]:
drv.close()
print('disconnected')